In [10]:
import pandas as pd
import numpy as np


In [11]:
df = pd.read_csv("transactions.csv", parse_dates=["date"])
print(df.head())
print("Nombre de transactions :")
print(len(df))


        date    client   montant devise pays types_de_paiement
0 2020-04-03   BFI-LUX  16834.87    JPY   DE          VIREMENT
1 2020-04-20    TER-DE   4825.89    EUR   JP            CHEQUE
2 2020-09-02  DASSAULT   2059.65    CHF   CA    PAIEMENT_CARTE
3 2020-12-20  BTA-CORP   9657.99    USD   DE          VIREMENT
4 2020-01-20  DASSAULT   2046.98    USD   FR    PAIEMENT_CARTE
Nombre de transactions :
2005


In [12]:
# DETECTION ET SUPPRESSION DES ANOMALIES

# Chaque règle est évaluée sur le dataframe original, avant toute suppression.
montants_eleves = df["montant"] >= 100000
dates_incoherentes = df["date"].dt.year > 2020
devises_invalides = ~df["devise"].isin(["EUR", "USD", "GBP", "CHF", "JPY", "CAD", "BTC"])
montants_nuls = df["montant"] <= 0
doublons = df.duplicated(keep=False)

print("=== Anomalies détectées ===")
print(f"Montants élevés : {montants_eleves.sum()}")
print(df.loc[montants_eleves, ["date", "client", "montant", "devise"]].to_string(index=False))

print(f"\nDates incohérentes : {dates_incoherentes.sum()}")
print(df.loc[dates_incoherentes, ["date", "client", "montant", "devise"]].to_string(index=False))

print(f"\nDevises invalides : {devises_invalides.sum()}")
print(df.loc[devises_invalides, ["date", "client", "montant", "devise"]].to_string(index=False))

print(f"\nMontants nuls ou négatifs : {montants_nuls.sum()}")
print(df.loc[montants_nuls, ["date", "client", "montant", "devise"]].to_string(index=False))

print(f"\nLignes appartenant à un doublon : {doublons.sum()}")
print(df.loc[doublons, ["date", "client", "montant", "devise", "pays", "types_de_paiement"]].to_string(index=False))

# Une ligne peut respecter plusieurs règles : on utilise l'union des masques.
anomalies = montants_eleves | dates_incoherentes | devises_invalides | montants_nuls | doublons
print(f"\nTotal de lignes supprimées : {anomalies.sum()}")

df = df.loc[~anomalies].copy()
print(f"Transactions restantes : {len(df)}")

=== Anomalies détectées ===
Montants élevés : 5
      date  client   montant devise
2020-05-27 ACME-FR 875541.78    BTC
2020-07-30  THALES 407382.17    JPY
2020-11-25 CITROEN 546097.11    BTC
2020-02-10  TER-DE 828049.80    BTC
2020-10-28  TER-DE 708593.54    USD

Dates incohérentes : 5
      date   client  montant devise
2025-07-13 BTA-CORP  1953.72    BTC
2025-12-11   TER-DE   931.14    JPY
2025-05-16  ACME-FR  1165.34    JPY
2025-02-13   TER-DE  4271.36    CHF
2025-12-05  CITROEN  7123.96    BTC

Devises invalides : 5
      date   client  montant devise
2020-01-26  CITROEN  1143.37    DEF
2020-08-14  BFI-LUX  1356.80    ABC
2020-06-12  CITROEN  1031.26    ABC
2020-05-21 BTA-CORP  2473.38    ABC
2020-11-26   THALES  2707.22    DEF

Montants nuls ou négatifs : 5
      date  client  montant devise
2020-09-01  TER-DE      0.0    EUR
2020-01-23 CITROEN      0.0    BTC
2020-01-17 ACME-FR      0.0    USD
2020-08-26 CITROEN      0.0    EUR
2020-06-02 CITROEN      0.0    BTC

Lignes apparten

In [13]:
#Clear the data : its over a period of a year , so we can add a column for the month 
# of the transaction to see if there is any seasonality in the data
df["mois"] = df["date"].dt.month
print(df.head())

        date    client   montant devise pays types_de_paiement  mois
0 2020-04-03   BFI-LUX  16834.87    JPY   DE          VIREMENT     4
1 2020-04-20    TER-DE   4825.89    EUR   JP            CHEQUE     4
2 2020-09-02  DASSAULT   2059.65    CHF   CA    PAIEMENT_CARTE     9
3 2020-12-20  BTA-CORP   9657.99    USD   DE          VIREMENT    12
4 2020-01-20  DASSAULT   2046.98    USD   FR    PAIEMENT_CARTE     1


In [14]:
# Analyse par clients
par_clients = df.groupby("client").agg(
    nombre_transactions=("client", "size"),
    montant_total=("montant", "sum"),
    montant_moyen=("montant", "mean"),
    montant_max=("montant", "max"),
    montant_min=("montant", "min")
).reset_index()
print(par_clients.head())

# Analyse par mois
par_mois = df.groupby("mois").agg(
    nombre_transactions=("client", "size"),
    montant_total=("montant", "sum"),
    montant_moyen=("montant", "mean"),
    montant_max=("montant", "max"),
    montant_min=("montant", "min")
).reset_index()
print(par_mois)

     client  nombre_transactions  montant_total  montant_moyen  montant_max  \
0   ACME-FR                  299     1454179.15    4863.475418     25858.52   
1   BFI-LUX                  298     1529108.14    5131.235369     27114.87   
2  BTA-CORP                  271     1280938.13    4726.708967     27052.11   
3   CITROEN                  262     1442668.74    5506.369237     35446.81   
4  DASSAULT                  288     1424699.50    4946.873264     40690.22   

   montant_min  
0        14.25  
1        25.17  
2        13.45  
3        79.34  
4        10.37  
    mois  nombre_transactions  montant_total  montant_moyen  montant_max  \
0      1                  181      775806.11    4286.221602     22145.96   
1      2                  158      935054.09    5918.063861     40690.22   
2      3                  153      723397.23    4728.086471     25233.60   
3      4                  157      760088.37    4841.327197     35446.81   
4      5                  161      813212.3

AMELIORATION : automatiser un rapport quotidien (export Excel/PDF)
 et ajouter une visualisation (graphiques matplotlib) du volume
 par client et par mois, pour un suivi en un coup d'œil.

In [15]:
par_mois = df.groupby("mois")["montant"].sum()
print("\n=== Volume par mois ===")
print(par_mois.round(2))


=== Volume par mois ===
mois
1      775806.11
2      935054.09
3      723397.23
4      760088.37
5      813212.37
6      798480.33
7      689848.11
8      898119.85
9      805391.48
10     767947.96
11     870031.62
12    1012594.86
Name: montant, dtype: float64


In [16]:
# DETECTION STATISTIQUE COMPLEMENTAIRE

# Le Z-score repère les montants statistiquement atypiques restants.
moyenne = df["montant"].mean()
ecart_type = df["montant"].std()
df["z_score"] = (df["montant"] - moyenne) / ecart_type

anomalies_zscore = df[df["z_score"].abs() > 3].sort_values("montant", ascending=False)

print(f"\n=== Anomalies statistiques restantes : {len(anomalies_zscore)} ===")
resultats_anomalies = anomalies_zscore[["date", "client", "montant", "devise", "z_score"]].copy()
resultats_anomalies["montant"] = resultats_anomalies["montant"].round(2)
resultats_anomalies["z_score"] = resultats_anomalies["z_score"].round(2)
print(resultats_anomalies.to_string(index=False))

# Suppression des anomalies Z-score après affichage.
df = df.loc[~df.index.isin(anomalies_zscore.index)].drop(columns="z_score")
print(f"Anomalies statistiques supprimées : {len(anomalies_zscore)}")
print(f"Transactions finales : {len(df)}")


=== Anomalies statistiques restantes : 38 ===
      date   client  montant devise  z_score
2020-02-25 DASSAULT 40690.22    CHF     7.30
2020-04-21  CITROEN 35446.81    BTC     6.23
2020-06-17   TER-DE 30970.65    CAD     5.31
2020-07-05   TER-DE 29552.05    EUR     5.02
2020-06-22  CITROEN 28267.12    BTC     4.76
2020-11-28  BFI-LUX 27114.87    CAD     4.53
2020-02-06 BTA-CORP 27052.11    GBP     4.51
2020-12-29   THALES 27031.69    CHF     4.51
2020-09-22  ACME-FR 25858.52    JPY     4.27
2020-06-11   THALES 25480.50    GBP     4.19
2020-03-01   TER-DE 25233.60    JPY     4.14
2020-05-11  CITROEN 25116.55    USD     4.12
2020-11-16  CITROEN 24310.74    CHF     3.95
2020-12-01   THALES 24133.24    CAD     3.92
2020-05-22  BFI-LUX 24099.89    GBP     3.91
2020-03-03   TER-DE 23839.39    GBP     3.86
2020-02-20  CITROEN 23623.89    GBP     3.81
2020-09-01  ACME-FR 23225.75    GBP     3.73
2020-01-09   THALES 22145.96    EUR     3.51
2020-10-04 BTA-CORP 21849.22    JPY     3.45
2020-08-

 AMELIORATIONS POSSIBLES (pistes pour aller plus loin) :

 1. Machine Learning : remplacer les règles fixes par un modèle
    de détection d'anomalies non supervisé (ex : Isolation Forest),
    capable de repérer des anomalies plus complexes qu'un simple
    seuil, et qui s'adapte aux habitudes de chaque client.

 2. Détection par client : calculer un Z-score PAR client plutôt
    que global, car un montant "normal" pour Thales ne l'est pas
    forcément pour un petit client.

aussi : AMELIORATION : connecter ces analyses à une vraie source de données
 (base SQL ou API) au lieu d'un CSV statique, pour traiter les flux
 de paiement en continu.